# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
orders.head()

  id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
0   order_0  user_6993        2025-05-22  Argentina     desktop   
1   order_1  user_1329        2025-06-15     Mexico     desktop   
2   order_2  user_3194        2025-05-02  Argentina     desktop   
3   order_3  user_4510        2025-06-09   Colombia      mobile   
4   order_4  user_5044        2025-03-30  Argentina     desktop   

  fuente_referencia       nombre_producto categoria_producto  cantidad  \
0           organic       Jacket-Winter-M               Moda       2.0   
1       paid_search  Tablet-Standard-64GB        Electronica       1.0   
2            social        Blender-XL-Red              Hogar       2.0   
3            social  Tablet-Standard-64GB        Electronica       1.0   
4       paid_search        Blender-XL-Red              Hogar       1.0   

   precio_unitario  monto_descuento  monto_total  
0           332.69              0.0       665.37  
1           176.86              5.0       171.86  

In [4]:
catalog.head()

        nombre_producto categoria_producto  costo_unitario  \
0    Laptop-Gaming-16GB        Electrónica          280.68   
1       Phone-Pro-128GB        Electrónica           10.12   
2  Tablet-Standard-64GB        Electrónica           25.21   
3        Blender-XL-Red              Hogar          176.64   
4      Vacuum-Pro-Black              Hogar           16.60   

                 proveedor  
0   Fuller, Pena and Myers  
1                 King Ltd  
2               Bowers LLC  
3                Long-Reid  
4  Rivera, Carr and Finley  

In [5]:
marketing.head()

        fecha      pais            id_campaña        canal    gasto
0  2025-01-01    Mexico        organic_Mexico      organic  2446.25
1  2025-01-01    Mexico    paid_search_Mexico  paid_search  2704.34
2  2025-01-01    Mexico         social_Mexico       social  2045.01
3  2025-01-01  Colombia      organic_Colombia      organic  2597.21
4  2025-01-01  Colombia  paid_search_Colombia  paid_search  1771.40

---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 



---

In [6]:
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'], errors='coerce')
marketing['fecha'] = pd.to_datetime(marketing['fecha'], errors='coerce')
orders.info()
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           25100 non-null  object        
 1   id_usuario          25100 non-null  object        
 2   fecha_hora_pedido   25100 non-null  datetime64[ns]
 3   pais                24800 non-null  object        
 4   dispositivo         25080 non-null  object        
 5   fuente_referencia   25070 non-null  object        
 6   nombre_producto     25070 non-null  object        
 7   categoria_producto  25020 non-null  object        
 8   cantidad            25050 non-null  float64       
 9   precio_unitario     25050 non-null  float64       
 10  monto_descuento     25050 non-null  float64       
 11  monto_total         25100 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.3+ MB
<class 'pandas.core.frame.DataFrame'

In [7]:
orders = orders.drop_duplicates(subset=['id_pedido'], keep='first')

In [8]:
subtotal_calculado = orders['precio_unitario']* orders['cantidad']
orders['diferencia'] = subtotal_calculado - orders['monto_total']
print (subtotal_calculado)
print(orders['diferencia'])

0        665.38
1        176.86
2        205.98
3        257.87
4        336.28
          ...  
24995     64.57
24996    585.80
24997    226.85
24998    836.12
24999    456.46
Length: 25000, dtype: float64
0         0.01
1         5.00
2         9.99
3        15.00
4         0.00
         ...  
24995     0.00
24996    14.99
24997     0.00
24998     0.00
24999     5.00
Name: diferencia, Length: 25000, dtype: float64


In [9]:

orders['coincide'] = abs(orders['diferencia'] - orders['monto_descuento']) <= 0.01
orders['coincide'].value_counts()


True     23805
False     1195
Name: coincide, dtype: int64

In [10]:
sin_coincidencia = orders[orders['coincide'] == False]
sin_coincidencia.head(15)

   id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
2    order_2  user_3194        2025-05-02  Argentina     desktop   
24  order_24   user_458        2025-01-05   Colombia         NaN   
35  order_35  user_3817        2025-02-08     Mexico         NaN   
41  order_41  user_2963        2025-01-27     Mexico         NaN   
74  order_74  user_6172        2025-01-15  Argentina     desktop   
75  order_75  user_6588        2025-06-19   Colombia      mobile   
76  order_76  user_3193        2025-03-17  Argentina      mobile   
77  order_77   user_775        2025-06-24  Argentina     desktop   
78  order_78  user_2702        2025-03-19  Argentina     desktop   
79  order_79  user_6438        2025-03-22   Colombia     desktop   
80  order_80  user_6790        2025-01-03     Mexico     desktop   
81  order_81   user_232        2025-05-16     Mexico      mobile   
82  order_82  user_5665        2025-02-07   Colombia      mobile   
83  order_83  user_5225        2025-04-25  Argen

In [11]:
orders = orders.dropna(subset=['cantidad', 'precio_unitario', 'monto_total'])

In [12]:
print(orders.isna().sum())
print(orders.isna().mean())

id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad                0
precio_unitario         0
monto_descuento         0
monto_total             0
diferencia              0
coincide                0
dtype: int64
id_pedido             0.000000
id_usuario            0.000000
fecha_hora_pedido     0.000000
pais                  0.012024
dispositivo           0.000802
fuente_referencia     0.001202
nombre_producto       0.001202
categoria_producto    0.001202
cantidad              0.000000
precio_unitario       0.000000
monto_descuento       0.000000
monto_total           0.000000
diferencia            0.000000
coincide              0.000000
dtype: float64


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [13]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [14]:
# tu código aquí
revenue_registrado = orders['monto_total'].sum()
orders['monto_calculado'] = (orders['cantidad'].abs() * orders['precio_unitario'].abs()) - orders['monto_descuento'].abs()
revenue_recalculado = orders['monto_calculado'].sum()

print(f"Revenue registrado (de duplicado): ${revenue_registrado:,.2f}")
print(f"Revenue corregido (consistente): ${revenue_recalculado:,.2f}")

Revenue registrado (de duplicado): $51,965,834.26
Revenue corregido (consistente): $51,968,129.67


In [15]:
df_completo = orders.merge(
    catalog[['nombre_producto', 'costo_unitario', 'categoria_producto']], 
    on='nombre_producto', 
    how='left'
)

df_completo['cogs_pedido'] = df_completo['cantidad'] * df_completo['costo_unitario']
df_completo['ganancia_bruta_pedido'] = df_completo['monto_total'] - df_completo['cogs_pedido']

revenue_total = df_completo['monto_total'].sum()
cogs_total = df_completo['cogs_pedido'].sum()
marketing_total = marketing['gasto'].sum()

ganancia_bruta = revenue_total - cogs_total
ganancia_neta = ganancia_bruta - marketing_total
margen_neta_pct = (ganancia_neta / revenue_total) * 100

print(f"Revenue Total:         ${revenue_total:,.2f}")
print(f"COGS Total:            ${cogs_total:,.2f}")
print(f"Ganancia Bruta:        ${ganancia_bruta:,.2f}")
print(f"Gasto Marketing:       ${marketing_total:,.2f}")
print(f"Profit:                ${ganancia_neta:,.2f}")
print(f"Margen Neto:           {margen_neta_pct:.2f}%")

Revenue Total:         $51,965,834.26
COGS Total:            $43,124,018.41
Ganancia Bruta:        $8,841,815.85
Gasto Marketing:       $2,871,843.53
Profit:                $5,969,972.32
Margen Neto:           11.49%


In [16]:
ticket_promedio = orders['monto_total'].sum() / orders['id_pedido'].nunique()
print(f"Ticket Promedio por Orden: ${ticket_promedio:,.2f}")

Ticket Promedio por Orden: $2,082.80


In [17]:
promedio_productos = orders['cantidad'].mean()
mediana_productos = orders['cantidad'].median()
print(f"Cantidad promedio por orden: {promedio_productos:.2f} unidades")
print(f"Mediana por orden: {mediana_productos:.0f} unidades")

Cantidad promedio por orden: 7.12 unidades
Mediana por orden: 2 unidades


In [18]:
top_unidades = orders.groupby('nombre_producto')['cantidad'].sum().sort_values(ascending=False)
top_ingresos = orders.groupby('nombre_producto')['monto_total'].sum().sort_values(ascending=False)
top_frecuencia = orders.groupby('nombre_producto')['id_pedido'].nunique().sort_values(ascending=False)

print("Producto más vendido por unidades:", top_unidades.idxmax(), f"({top_unidades.max():,.0f} unidades)")
print("Producto más vendido por ingresos:", top_ingresos.idxmax(), f"(${top_ingresos.max():,.2f})")
print("Producto más vendido por frecuencia:", top_frecuencia.idxmax(), f"({top_frecuencia.max():,} órdenes)")

Producto más vendido por unidades: Laptop-Gaming-16GB (144,198 unidades)
Producto más vendido por ingresos: Laptop-Gaming-16GB ($43,429,372.98)
Producto más vendido por frecuencia: Blender-XL-Red (4,176 órdenes)


In [19]:
gasto_por_canal = marketing.groupby('canal')['gasto'].sum().sort_values(ascending=False).reset_index()
gasto_por_canal['porcentaje'] = (gasto_por_canal['gasto'] / gasto_por_canal['gasto'].sum()) * 100

print(gasto_por_canal.to_string(index=False))

      canal     gasto  porcentaje
     social 918043.21   34.068925
    organic 913533.01   33.901550
paid_search 863088.21   32.029525


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [20]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [21]:
# Explorar tabla events
# =========================
query_events = '''
SELECT 
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY 
    CASE nombre_evento
        WHEN 'page_view' THEN 1         
        WHEN 'view_item' THEN 2         
        WHEN 'add_to_cart' THEN 3       
        WHEN 'begin_checkout' THEN 4   
        WHEN 'purchase' THEN 5          
        ELSE 6
    END;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

      nombre_evento  usuarios_unicos
0       add_to_cart             7634
1    begin_checkout             7208
2          purchase             6240
3  add_payment_info             6250
4       first_visit             7796

In [22]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
WITH funnel_base AS (
    SELECT 
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    GROUP BY nombre_evento
),
funnel_ordenado AS (
    SELECT 
        nombre_evento,
        usuarios_unicos,
        CASE nombre_evento
            WHEN 'page_view' THEN 1         
            WHEN 'view_item' THEN 2         
            WHEN 'add_to_cart' THEN 3       
            WHEN 'begin_checkout' THEN 4   
            WHEN 'purchase' THEN 5          
            ELSE 6
        END AS paso
    FROM funnel_base
)
SELECT 
    nombre_evento,
    usuarios_unicos,
    -- Porcentaje de conversión respecto a la primera etapa (page_view)
    ROUND(
        (usuarios_unicos::numeric / FIRST_VALUE(usuarios_unicos) OVER(ORDER BY paso)) * 100, 2
    ) AS pct_conversion_total,
    -- Porcentaje de usuarios retendidos respecto al paso anterior
    ROUND(
        (usuarios_unicos::numeric / LAG(usuarios_unicos, 1, usuarios_unicos) OVER(ORDER BY paso)) * 100, 2
    ) AS pct_paso_anterior,
    -- Porcentaje de pérdida (drop-off) respecto al paso anterior
    ROUND(
        100 - ((usuarios_unicos::numeric / LAG(usuarios_unicos, 1, usuarios_unicos) OVER(ORDER BY paso)) * 100), 2
    ) AS pct_drop_off
FROM funnel_ordenado
ORDER BY paso;
'''
totals = pd.read_sql(query_totals, con=engine)
totals

      nombre_evento  usuarios_unicos  pct_conversion_total  pct_paso_anterior  \
0       add_to_cart             7634                100.00             100.00   
1    begin_checkout             7208                 94.42              94.42   
2          purchase             6240                 81.74              86.57   
3  add_payment_info             6250                 81.87             100.16   
4       first_visit             7796                102.12             124.74   
5       select_item             7582                 99.32              97.26   

   pct_drop_off  
0          0.00  
1          5.58  
2         13.43  
3         -0.16  
4        -24.74  
5          2.74  

In [23]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel_counts AS (
    SELECT 
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos,
        CASE nombre_evento
            WHEN 'page_view' THEN 1         
            WHEN 'view_item' THEN 2         
            WHEN 'add_to_cart' THEN 3       
            WHEN 'begin_checkout' THEN 4   
            WHEN 'purchase' THEN 5          
            ELSE 6
        END AS paso
    FROM events
    GROUP BY nombre_evento
),
funnel_con_lag AS (
    SELECT 
        paso,
        nombre_evento,
        usuarios_unicos,
        LAG(usuarios_unicos) OVER(ORDER BY paso) AS usuarios_paso_anterior,
        FIRST_VALUE(usuarios_unicos) OVER(ORDER BY paso) AS usuarios_iniciales
    FROM funnel_counts
)
SELECT 
    paso,
    nombre_evento,
    usuarios_unicos,
    -- Tasa de conversión paso a paso (respecto a la etapa inmediatamente anterior)
    COALESCE(
        ROUND((usuarios_unicos::numeric / usuarios_paso_anterior) * 100, 2), 
        100.00
    ) AS tasa_conversion_paso_a_paso,
    -- Tasa de conversión global (respecto a la etapa inicial page_view)
    ROUND((usuarios_unicos::numeric / usuarios_iniciales) * 100, 2) AS tasa_conversion_global
FROM funnel_con_lag
ORDER BY paso;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

   paso     nombre_evento  usuarios_unicos  tasa_conversion_paso_a_paso  \
0     3       add_to_cart             7634                       100.00   
1     4    begin_checkout             7208                        94.42   
2     5          purchase             6240                        86.57   
3     6  add_payment_info             6250                       100.16   
4     6       first_visit             7796                       124.74   
5     6       select_item             7582                        97.26   

   tasa_conversion_global  
0                  100.00  
1                   94.42  
2                   81.74  
3                   81.87  
4                  102.12  
5                   99.32  

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [24]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *,
CAST(fecha_registro AS DATE)
FROM users
;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

  id_usuario fecha_registro       país dispositivo tipo_plan fecha_registro
0     user_0     2025-01-29     Mexico      mobile      free     2025-01-29
1     user_1     2025-01-07     Mexico      mobile      free     2025-01-07
2     user_2     2025-03-12  Argentina      mobile      free     2025-03-12

In [25]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *,
CAST(fecha_actividad AS DATE)
FROM user_activity
;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

  id_usuario fecha_actividad  dias_despues_registro  activo fecha_actividad
0     user_0      2025-02-05                      7       0      2025-02-05
1     user_0      2025-02-12                     14       1      2025-02-12
2     user_0      2025-02-19                     21       1      2025-02-19

In [26]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH primer_actividad AS (
SELECT 
      id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS mes_cohorte
    FROM users
    WHERE fecha_registro IS NOT NULL
    GROUP BY id_usuario, CAST(fecha_registro AS DATE)
),
actividad_mensual AS (
SELECT DISTINCT
        a.id_usuario,
        p.mes_cohorte,
        (EXTRACT(YEAR FROM DATE_TRUNC('month', CAST(a.fecha_actividad AS DATE))) - EXTRACT(YEAR FROM p.mes_cohorte)) * 12 +
        (EXTRACT(MONTH FROM DATE_TRUNC('month', CAST(a.fecha_actividad AS DATE))) - EXTRACT(MONTH FROM p.mes_cohorte)) AS periodo_mes
    FROM user_activity a
    JOIN primer_actividad p ON a.id_usuario = p.id_usuario
),
conteo_retencion AS (
SELECT
TO_CHAR(mes_cohorte, 'YYYY-MM') AS cohorte,
        periodo_mes,
        COUNT(DISTINCT id_usuario) AS usuarios_activos
    FROM actividad_mensual
    GROUP BY mes_cohorte, periodo_mes
),
tamano_cohorte AS (
SELECT 
        cohorte,
        usuarios_activos AS total_inicial
    FROM conteo_retencion
    WHERE periodo_mes = 0
)
SELECT 
    c.cohorte,
    c.periodo_mes,
    c.usuarios_activos,
    t.total_inicial,
    ROUND((c.usuarios_activos::numeric / t.total_inicial) * 100, 2) AS pct_retencion
FROM conteo_retencion c
JOIN tamano_cohorte t ON c.cohorte = t.cohorte
ORDER BY c.cohorte ASC, c.periodo_mes ASC;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

   cohorte  periodo_mes  usuarios_activos  total_inicial  pct_retencion
0  2025-01          0.0              1239           1239         100.00
1  2025-01          1.0              1486           1239         119.94
2  2025-02          0.0              1104           1104         100.00
3  2025-02          1.0              1444           1104         130.80
4  2025-03          0.0              1285           1285         100.00
5  2025-03          1.0              1477           1285         114.94
6  2025-04          0.0              1212           1212         100.00
7  2025-04          1.0              1508           1212         124.42
8  2025-05          0.0              1299           1299         100.00
9  2025-05          1.0              1516           1299         116.71

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** No existe diferencia estadísticamente significativa en la tasa de conversión entre la interfaz actual y la nueva UI del checkout. no se rechaza la hipótesis nula (H₀). La diferencia observada de 0.6% entre grupos no es estadísticamente significativa y se debe al azar, por lo que no hay evidencia suficiente de que la nueva UI incremente la conversión.
   - **H₁ (Hipótesis alternativa):** Existe una diferencia estadísticamente significativa en la tasa de conversión entre ambos grupos
   
**Test estadístico:** Prueba Z de dos muestras para proporciones
Estadístico Z: -0.8133
p-value:      0.41606
**Nivel de significancia alpha:** 0.05 (95% de confianza)

In [27]:
# tu código aquí
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency

In [28]:
df = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')

In [29]:
df.head(5)

   id_usuario     variante  convirtio dispositivo       pais  duracion_sesion  \
0  exp_user_0  tratamiento          0      mobile  Argentina           114.41   
1  exp_user_1  tratamiento          0     desktop     Mexico           170.03   
2  exp_user_2      control          1      mobile   Colombia           140.21   
3  exp_user_3  tratamiento          0      mobile   Colombia           151.45   
4  exp_user_4  tratamiento          0     desktop     Mexico           299.96   

    timestamp  
0  2025-03-28  
1  2025-01-15  
2  2025-03-18  
3  2025-06-03  
4  2025-01-12  

In [30]:
resumen = df.groupby('variante')['convirtio'].agg(['count', 'sum', 'mean']).reset_index()
resumen.columns = ['Grupo', 'Total_Usuarios', 'Conversiones', 'Tasa_Conversion']

print("Resumen por Grupo")
print(resumen)

Resumen por Grupo
         Grupo  Total_Usuarios  Conversiones  Tasa_Conversion
0      control            4965           779         0.156898
1  tratamiento            5035           820         0.162860


In [31]:
conversi_exitos = resumen['Conversiones'].values
total_observaciones = resumen['Total_Usuarios'].values

z_stat, p_value = proportions_ztest(count=conversi_exitos, nobs=total_observaciones)

print(f"\nResultados del Test Estadístico")
print(f"Estadístico Z: {z_stat:.4f}") 
print(f"p-value:      {p_value:.5f}")

alpha = 0.05
if p_value < alpha:
    print(f"\nResultado: Rechazamos la hipótesis nula (H0).")
    print("Conclusión: Existe una diferencia estadísticamente significativa en la tasa de conversión debido al cambio de UI.")
else:
    print(f"\nResultado: No se rechaza la hipótesis nula (H0).")
    print("Conclusión: No hay suficiente evidencia estadística para afirmar que la nueva UI impacte la tasa de conversión.")


Resultados del Test Estadístico
Estadístico Z: -0.8133
p-value:      0.41606

Resultado: No se rechaza la hipótesis nula (H0).
Conclusión: No hay suficiente evidencia estadística para afirmar que la nueva UI impacte la tasa de conversión.


---

In [2]:
print ('https://public.tableau.com/views/Project_17885024557020/Dashboard2?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link')

https://public.tableau.com/views/Project_17885024557020/Dashboard2?:language=en-US&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link
